# MNIST Digit Classification
A beginner-friendly, Google Colab-ready workflow: download MNIST, prepare training/testing datasets, train a neural-network classifier, and evaluate its test error.

**Metric:** test error rate = number of incorrect predictions / total test images.

## 1. Import libraries
TensorFlow includes Keras and the MNIST dataset loader. Matplotlib and NumPy are used for visualization and evaluation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
tf.keras.utils.set_random_seed(42)

## 2. Download MNIST and create the datasets
`load_data()` downloads MNIST the first time it runs and returns the official split: 60,000 training images and 10,000 test images. Pixel values are normalized from 0–255 to 0–1.

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", x_test.shape)
print("Test labels:", y_test.shape)

## 3. Inspect a few training examples
Each image is 28 × 28 grayscale pixels. The label is the digit (0–9).

In [ ]:
plt.figure(figsize=(8, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(f"Label: {y_train[i]}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## 4. Build a simple classifier
The model flattens each 28 × 28 image into 784 values, learns features in a 128-neuron ReLU layer, and outputs probabilities for the 10 digit classes.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 5. Train the model
The model trains on the training dataset. Validation data is taken from the training set; the test set remains untouched until final evaluation.

In [ ]:
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

## 6. Evaluate test performance
Evaluate once on the held-out test dataset. The test error rate is `1 - test accuracy`; the misclassification count is calculated from individual predictions.

In [ ]:
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
probabilities = model.predict(x_test, verbose=0)
y_pred = np.argmax(probabilities, axis=1)

incorrect = int(np.sum(y_pred != y_test))
total = int(len(y_test))
test_error_rate = incorrect / total

print(f"Test loss:             {test_loss:.4f}")
print(f"Test accuracy:         {test_accuracy:.4%}")
print(f"Test error rate:       {test_error_rate:.4%}")
print(f"Misclassified images:  {incorrect} / {total}")

## 7. Visualize training history
These plots show how training and validation accuracy/loss changed across epochs.

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Loss")

plt.tight_layout()
plt.show()

## 8. Inspect sample predictions
Green titles indicate correct predictions; red titles indicate mistakes.

In [ ]:
plt.figure(figsize=(10, 5))
for i in range(15):
    plt.subplot(3, 5, i + 1)
    plt.imshow(x_test[i], cmap="gray")
    color = "green" if y_pred[i] == y_test[i] else "red"
    plt.title(f"Pred: {y_pred[i]} | True: {y_test[i]}", color=color)
    plt.axis("off")
plt.tight_layout()
plt.show()

## Conclusion
The final test error rate and number of incorrect predictions are printed in Section 6. Results may vary slightly if the model architecture, random seed, or training settings are changed.

**Important:** The test dataset is held out from training and validation, so it provides an independent estimate of performance on unseen MNIST images.